In [0]:
#Break from this cell to new Gold layer notebook.
# Reading parquet & create dataframe. 

from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import os
BASE_DIR = '/Volumes/debora_ryan_susheela_hhs/default/upload_volume'
SILVER_PARQUET_DIR = os.path.join(BASE_DIR, "silver", "data", "age")
filepath_rates = os.path.join(SILVER_PARQUET_DIR, 'non_smokers')

def read_parquet(filepath: str) -> DataFrame:
    data_f = spark.read.parquet(filepath)
    return data_f
    
df_rates_clean_age_non_smokers = read_parquet(filepath_rates)


In [0]:
from pyspark.sql import functions as F
plan_ids = [
    "16842FL0010002",
    "16842FL0010001",
    "97325SC0080001",
    "84966SC0120001"
]

df_16842FL0010002 = df_rates_clean_age_non_smokers.filter(F.col("PlanId") == "16842FL0010002")
df_16842FL0010001 = df_rates_clean_age_non_smokers.filter(F.col("PlanId") == "16842FL0010001")
df_97325SC0080001 = df_rates_clean_age_non_smokers.filter(F.col("PlanId") == "97325SC0080001")
df_84966SC0120001 = df_rates_clean_age_non_smokers.filter(F.col("PlanId") == "84966SC0120001")

display(df_16842FL0010002)
display(df_16842FL0010001)
display(df_97325SC0080001)
display(df_84966SC0120001)

In [0]:
def compute_avg_individual_rate(df):
    return df.groupBy("Age").agg(
        F.avg(F.col("IndividualRate").cast("double")).alias("avg_IndividualRate")
    )

df_avg_individual_rate_16842FL0010002 = compute_avg_individual_rate(df_16842FL0010002)
df_avg_individual_rate_16842FL0010001 = compute_avg_individual_rate(df_16842FL0010001)
df_avg_individual_rate_97325SC0080001 = compute_avg_individual_rate(df_97325SC0080001)
df_avg_individual_rate_84966SC0120001 = compute_avg_individual_rate(df_84966SC0120001)

display(df_avg_individual_rate_16842FL0010002.orderBy("Age"))
display(df_avg_individual_rate_16842FL0010001.orderBy("Age"))
display(df_avg_individual_rate_97325SC0080001.orderBy("Age"))
display(df_avg_individual_rate_84966SC0120001.orderBy("Age"))

In [0]:
# Write data to parquet
def write(input_df: DataFrame, out_dir):
    return input_df.write.mode('overwrite').parquet(out_dir)

In [0]:
GOLD_PARQUET_DIR = os.path.join(BASE_DIR, "gold", "data", "age")
os.makedirs(GOLD_PARQUET_DIR, exist_ok=True)

write(df_avg_individual_rate_16842FL0010002, f"{GOLD_PARQUET_DIR}/16842FL0010002")
write(df_avg_individual_rate_16842FL0010001, f"{GOLD_PARQUET_DIR}/16842FL0010001")
write(df_avg_individual_rate_97325SC0080001, f"{GOLD_PARQUET_DIR}/97325SC0080001")
write(df_avg_individual_rate_84966SC0120001, f"{GOLD_PARQUET_DIR}/84966SC0120001")

In [0]:
import matplotlib.pyplot as plt

dfs = [
    ("16842FL0010002", df_avg_individual_rate_16842FL0010002),
    ("16842FL0010001", df_avg_individual_rate_16842FL0010001),
    ("97325SC0080001", df_avg_individual_rate_97325SC0080001),
    ("84966SC0120001", df_avg_individual_rate_84966SC0120001)
]

fig, axs = plt.subplots(2, 2, figsize=(16, 12))
axs = axs.flatten()

for i, (plan_id, df) in enumerate(dfs):
    pdf = df.orderBy("Age").toPandas()
    ax = axs[i]
    ax.plot(pdf["Age"], pdf["avg_IndividualRate"], marker="o", color="steelblue", linewidth=2)
    ax.set_xlabel("Age", fontsize=12)
    ax.set_ylabel("Average Individual Rate", fontsize=12)
    ax.set_title(f"Average Individual Rate by Age\nPlanId: {plan_id}", fontsize=14)
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    # Set x-ticks every 5 ticks
    xticks = ax.get_xticks()
    ax.set_xticks([tick for idx, tick in enumerate(xticks) if idx % 5 == 0])

plt.tight_layout()
plt.show()